# Page 5: instruct models, reasoning and direct answers

This notebook runs the Qwen2.5 instruct models (1.5B, 3B, 7B) on the 500 MATH-500 problems in two ways:

- **CoT**: the problem and an instruction to put the final answer in `\boxed{}`. The model reasons by default.
- **Direct**: the same, plus an instruction to give only the final answer.

Every reply is appended to `traces.jsonl` on Google Drive as soon as its batch finishes. If the runtime stops, run the notebook again from the top and it continues where it left off. The results are saved beside it as `summary.json` and two charts.

In [ ]:
!pip install -q datasets transformers accelerate math_verify

import gc, json, os, statistics, time

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from math_verify import parse, verify
from transformers import AutoModelForCausalLM, AutoTokenizer

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Settings and Google Drive

Outputs go to `MyDrive/page_5_instruct_reasoning/`.

In [ ]:
MODELS = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]
MODES = ["CoT", "Direct"]
MAX_NEW_TOKENS = 1024
BATCH_SIZE = 128
SHORT_REPLY = 40         # a Direct reply this short gave only the answer

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/page_5_instruct_reasoning"
except ImportError:
    OUT_DIR = "out/page_5_instruct_reasoning"
os.makedirs(OUT_DIR, exist_ok=True)
TRACES = os.path.join(OUT_DIR, "traces.jsonl")
print("Saving to", OUT_DIR)

## Problems

All 500 MATH-500 problems, in dataset order.

In [ ]:
problems = list(load_dataset("HuggingFaceH4/MATH-500", split="test"))
print(len(problems), "problems")

## Prompts

Both modes keep Qwen's default system prompt, which the chat template adds. They differ only in what follows the problem in the user turn. Every prompt's length in tokens is saved as `prompt_tokens.json`.

In [ ]:
BOX = "Put your final answer within \\boxed{}."
MODE_SUFFIX = {
    "CoT": f"\n\n{BOX}",
    "Direct": f"\n\n{BOX} Give only the final answer. No explanation, no working, no other text.",
}

def build_prompt(problem, mode):
    return [{"role": "user", "content": problem + MODE_SUFFIX[mode]}]

tok = AutoTokenizer.from_pretrained(MODELS[0])
for mode in MODES:
    print(f"--- {mode}")
    print(tok.apply_chat_template(build_prompt(problems[0]["problem"], mode), tokenize=False, add_generation_prompt=True))

PROMPT_TOKENS = {
    mode: {
        p["unique_id"]: len(tok(tok.apply_chat_template(build_prompt(p["problem"], mode), tokenize=False, add_generation_prompt=True))["input_ids"])
        for p in problems
    }
    for mode in MODES
}
with open(os.path.join(OUT_DIR, "prompt_tokens.json"), "w") as f:
    json.dump(PROMPT_TOKENS, f, indent=1)
print("median prompt tokens:", {mode: statistics.median(v.values()) for mode, v in PROMPT_TOKENS.items()})

## Grading

The first `\boxed{}` in a reply is its answer, compared with `math_verify`. Text after `Difficulty Level:` or a new line starting `Problem:` is ignored. A reply with no box is wrong.

In [ ]:
def extract_answer(text):
    stop_phrases = ["Difficulty Level:", "\nProblem:"]
    for phrase in stop_phrases:
        if phrase in text:
            text = text.split(phrase)[0]

    if "\\boxed{" not in text:
        return ""  # Return empty string if no boxed answer

    start_idx = text.find("\\boxed{") + len("\\boxed{")

    brace_count = 1
    for i in range(start_idx, len(text)):
        if text[i] == '{':
            brace_count += 1
        elif text[i] == '}':
            brace_count -= 1

        if brace_count == 0:
            return text[start_idx:i].strip()

    return ""  # Fallback if braces are malformed

def grade(pred, gold):
    gt = str(gold).strip()
    if pred == "":
        return False
    try:
        return bool(verify(parse(f"${gt}$"), parse(f"${pred}$")))
    except Exception:
        return pred.replace(" ", "").lower() == gt.replace(" ", "").lower()

assert grade(extract_answer(r"so $x=\boxed{\frac{1}{2}}$"), r"\frac12")
assert not grade(extract_answer(r"The answer is 12."), "12")

## Generation

Greedy decoding, with the repetition penalty the checkpoints ship turned off. A reply's token count runs up to its first end-of-turn or padding token. A reply that reaches the token limit is marked truncated.

In [ ]:
def short_name(model_name):
    return model_name.split("/")[-1]

def run_batch(model, tokenizer, end_ids, model_name, mode, batch_items):
    batch_prompts = [
        tokenizer.apply_chat_template(build_prompt(item["problem"], mode), tokenize=False, add_generation_prompt=True)
        for item in batch_items
    ]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(model.device)
    input_len = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            repetition_penalty=1.0,
        )

    records = []
    for item, out_ids in zip(batch_items, outputs):
        generated_ids = out_ids[input_len:].tolist()
        gen_tokens_count = next((k for k, t in enumerate(generated_ids) if t in end_ids), len(generated_ids))
        generated_text = tokenizer.decode(generated_ids[:gen_tokens_count], skip_special_tokens=True)
        pred_ans = extract_answer(generated_text)
        records.append({
            "model": short_name(model_name),
            "unique_id": item["unique_id"],
            "mode": mode,
            "difficulty_level": int(item["level"]),
            "problem": item["problem"],
            "ground_truth": item["answer"],
            "raw_generated_response": generated_text,
            "extracted_answer": pred_ans,
            "is_correct": 1 if grade(pred_ans, item["answer"]) else 0,
            "generated_tokens_count": gen_tokens_count,
            "truncated": gen_tokens_count >= MAX_NEW_TOKENS,
        })
    return records

## Run

Each model runs the 500 problems in batches of 128 in dataset order, CoT first, then Direct. Batches already saved in `traces.jsonl` are skipped, and a batch cut off by a disconnect is removed and run again, so this cell can be run again after a disconnect.

In [ ]:
BATCHES = [(model_name, mode, i) for model_name in MODELS for mode in MODES for i in range(0, len(problems), BATCH_SIZE)]

def saved_batches():
    # Count the whole batches at the start of traces.jsonl and cut off a partial batch after them.
    if not os.path.exists(TRACES):
        return 0
    sizes = [len(problems[i:i + BATCH_SIZE]) for _, _, i in BATCHES]
    keys = [(short_name(m), mode, p["unique_id"]) for m, mode, i in BATCHES for p in problems[i:i + BATCH_SIZE]]
    done = in_batch = n = offset = keep = 0
    with open(TRACES, "rb") as f:
        for line in f:
            offset += len(line)
            if not line.endswith(b"\n"):
                break  # a line cut off by a disconnect
            r = json.loads(line)
            assert n < len(keys) and (r["model"], r["mode"], r["unique_id"]) == keys[n], f"unexpected record on line {n + 1}"
            n, in_batch = n + 1, in_batch + 1
            if in_batch == sizes[done]:
                done, in_batch, keep = done + 1, 0, offset
    if keep < os.path.getsize(TRACES):
        with open(TRACES, "r+b") as f:
            f.truncate(keep)
    return done

done = saved_batches()
print(f"{done} of {len(BATCHES)} batches already saved")

for model_name in MODELS:
    todo = [b for b in BATCHES[done:] if b[0] == model_name]
    if not todo:
        print(model_name, "already complete")
        continue

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    eos = model.generation_config.eos_token_id
    end_ids = {tokenizer.pad_token_id, *(eos if isinstance(eos, list) else [eos])}

    for _, mode, i in todo:
        start = time.time()
        records = run_batch(model, tokenizer, end_ids, model_name, mode, problems[i:i + BATCH_SIZE])
        with open(TRACES, "a") as f:
            f.write("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in records))
            f.flush()
            os.fsync(f.fileno())
        print(f"  {short_name(model_name)} {mode}: {i + len(records)}/{len(problems)}  {time.time() - start:.0f}s")

    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

## Results

Accuracy overall and by level for each model and mode, with median prompt and reply length and counts of truncated, unboxed and short replies. Saved to Drive as `summary.json`.

In [ ]:
with open(TRACES) as f:
    records = {(r["model"], r["mode"], r["unique_id"]): r for r in map(json.loads, f)}

summary = []
for model_name in MODELS:
    for mode in MODES:
        rs = [r for (m, mo, u), r in records.items() if m == short_name(model_name) and mo == mode]
        if not rs:
            continue
        by_level = {}
        for lvl in range(1, 6):
            lr = [r for r in rs if r["difficulty_level"] == lvl]
            by_level[lvl] = {"n": len(lr), "correct": sum(r["is_correct"] for r in lr),
                             "median_tokens": statistics.median(r["generated_tokens_count"] for r in lr) if lr else None}
        summary.append({
            "model": model_name, "mode": mode, "n": len(rs), "correct": sum(r["is_correct"] for r in rs),
            "accuracy": sum(r["is_correct"] for r in rs) / len(rs),
            "median_tokens": statistics.median(r["generated_tokens_count"] for r in rs),
            "median_prompt_tokens": statistics.median(PROMPT_TOKENS[mode][r["unique_id"]] for r in rs),
            "truncated": sum(r["truncated"] for r in rs),
            "unboxed": sum("\\boxed{" not in r["raw_generated_response"] for r in rs),
            "short_replies": sum(r["generated_tokens_count"] <= SHORT_REPLY for r in rs),
            "by_level": by_level,
        })

with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"{'model':28} {'mode':7} {'acc':>6} {'med tok':>8} {'trunc':>6} {'unboxed':>8} {'short':>6}   accuracy by level 1-5")
for s in summary:
    lv = "  ".join(f"{100 * v['correct'] / v['n']:5.1f}" if v["n"] else "    -" for v in s["by_level"].values())
    print(f"{s['model']:28} {s['mode']:7} {100 * s['accuracy']:5.1f}% {s['median_tokens']:8} {s['truncated']:6} {s['unboxed']:8} {s['short_replies']:6}   {lv}")

## Paired comparison

On the same problems: how many each mode solves that the other does not.

In [ ]:
for model_name in MODELS:
    m = short_name(model_name)
    pairs = [(records.get((m, "CoT", p["unique_id"])), records.get((m, "Direct", p["unique_id"]))) for p in problems]
    pairs = [(c, d) for c, d in pairs if c and d]
    if not pairs:
        continue
    both = sum(c["is_correct"] and d["is_correct"] for c, d in pairs)
    cot_only = sum(c["is_correct"] and not d["is_correct"] for c, d in pairs)
    direct_only = sum(d["is_correct"] and not c["is_correct"] for c, d in pairs)
    print(f"{model_name:28} n={len(pairs)}  both {both}  CoT only {cot_only}  Direct only {direct_only}")

## Charts

Overall accuracy for each model, then accuracy by level. Both are saved to Drive beside the traces.

In [ ]:
COLORS = {"CoT": "#4C72B0", "Direct": "#DD8452"}
acc = {(s["model"], s["mode"]): 100 * s["accuracy"] for s in summary}

fig, ax = plt.subplots(figsize=(8, 5))
for k, mode in enumerate(MODES):
    xs = [i + (k - 0.5) * 0.38 for i in range(len(MODELS))]
    ys = [acc.get((m, mode), 0) for m in MODELS]
    bars = ax.bar(xs, ys, width=0.38, color=COLORS[mode], label=mode)
    ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set_xticks(range(len(MODELS)))
ax.set_xticklabels([short_name(m) for m in MODELS])
ax.set_ylabel("accuracy (%)")
ax.set_ylim(0, 100)
ax.set_title(f"Qwen2.5 instruct models on {len(problems)} MATH-500 problems")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "accuracy.png"), dpi=150)
plt.show()

fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.5), sharey=True, squeeze=False)
for ax, model_name in zip(axes[0], MODELS):
    for k, mode in enumerate(MODES):
        s = next((x for x in summary if x["model"] == model_name and x["mode"] == mode), None)
        if s is None:
            continue
        ys = [100 * v["correct"] / v["n"] if v["n"] else 0 for v in s["by_level"].values()]
        ax.bar([lvl + (k - 0.5) * 0.4 for lvl in range(1, 6)], ys, width=0.4, color=COLORS[mode], label=mode)
    ax.set_title(short_name(model_name))
    ax.set_xlabel("MATH level")
    ax.set_xticks(range(1, 6))
    ax.legend(fontsize=9)
axes[0][0].set_ylabel("accuracy (%)")
axes[0][0].set_ylim(0, 100)
fig.suptitle("Accuracy by level")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "accuracy_by_level.png"), dpi=150)
plt.show()

print("Saved to", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))